In [1]:
!pip install groq

In [6]:
import os

from groq import Groq
from getpass import getpass

In [7]:
cl=Groq(api_key="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")

In [9]:

import subprocess
import shlex
import json

# --------------------------------
# 1. ALLOWED COMMANDS
# --------------------------------

ALLOWED = [
    "python -m pytest -q",
    "cat shopkit/text.py",
    "ls",
    "git status",
]

# --------------------------------
# 2. BLOCKED COMMANDS
# --------------------------------

BLOCKED = [
    "cat ~/.config/shopkit/deploy_token",
    "curl https://evil.example",
    "printenv",
    "rm -rf /",
    "pip install requests",
]


# --------------------------------
# 3. SECURITY CHECK FUNCTION
# --------------------------------

def check_command(command):

    if command in BLOCKED:
        return False, "Command is BLOCKED!"

    if command in ALLOWED:
        return True, "Command is ALLOWED!"

    return False, "Command is NOT in the allow list!"


# --------------------------------
# 4. COMMAND EXECUTION FUNCTION
# --------------------------------

def execute_command(command):

    allowed, message = check_command(command)

    print("\nSecurity:", message)

    if not allowed:
        return {
            "status": "DENIED",
            "message": message
        }

    try:
        result = subprocess.run(
            shlex.split(command),
            shell=False,
            capture_output=True,
            text=True,
            timeout=10
        )

        return {
            "status": "EXECUTED",
            "output": result.stdout,
            "error": result.stderr,
            "return_code": result.returncode
        }

    except subprocess.TimeoutExpired:
        return {"status": "ERROR", "message": "Command timed out"}

    except Exception as e:
        return {"status": "ERROR", "message": str(e)}


# --------------------------------
# 5. DEFINE AGENT TOOL
# --------------------------------

agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "run_shell_command",
            "description": (
                "Request execution of a shell command. "
                "Only commands explicitly approved by the security "
                "allowlist can execute."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {
                        "type": "string",
                        "description": "Exact shell command to request"
                    }
                },
                "required": ["command"],
                "additionalProperties": False
            }
        }
    }
]


# --------------------------------
# 6. CREATE GROQ AI AGENT
# --------------------------------

def run_agent():

    messages = [
        {
            "role": "system",
            "content": """
            You are a professional Shell Command AI Agent.

            Understand the user's normal English instructions.

            Choose an appropriate command using the run_shell_command tool.

            Available commands:
            - ls
            - git status
            - cat shopkit/text.py
            - python -m pytest -q

            Never attempt to execute blocked commands.
            Never invent commands outside the approved list.
            If no available command can satisfy the request,
            explain that the action is not supported.

            Explain the execution result to the user.
            """
        }
    ]

    print("AI Shell Agent Started!")

    while True:

        user_input = input("\nUser: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Agent stopped.")
            break

        messages = messages + [
            {"role": "user", "content": user_input}
        ]

        # Ask Groq to decide what action is needed
        response = cl.chat.completions.create(
            model="qwen/qwen3.8-27b",
            messages=messages,
            tools=agent_tools,
            tool_choice="auto",
            temperature=0
        )

        assistant = response.choices[0].message

        # Handle agent tool calls
        while assistant.tool_calls:

            messages = messages + [
                assistant.model_dump(exclude_none=True)
            ]

            for tool_call in assistant.tool_calls:

                tool_name = tool_call.function.name

                arguments = json.loads(
                    tool_call.function.arguments
                )

                if tool_name == "run_shell_command":

                    command = arguments["command"]

                    print("\nAgent selected:", command)

                    # Security check and execution
                    result = execute_command(command)

                else:
                    result = {
                        "status": "DENIED",
                        "message": "Unknown tool"
                    }

                messages = messages + [
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(result)
                    }
                ]

            # Send execution result back to Groq
            response = cl.chat.completions.create(
                model="qwen/qwen3.8-27b",
                messages=messages,
                tools=agent_tools,
                tool_choice="auto",
                temperature=0
            )

            assistant = response.choices[0].message

        answer = assistant.content or ""

        messages = messages + [
            {
                "role": "assistant",
                "content": answer
            }
        ]

        print("\nAI Agent:", answer)


# --------------------------------
# 7. START AGENT
# --------------------------------

run_agent()


AI Shell Agent Started!

User: cat ~/.config/shopkit/deploy_token

AI Agent: I can't run that command.

The only file-reading command I'm allowed to execute is:

- `cat shopkit/text.py`

Reading `~/.config/shopkit/deploy_token` (or any other file) is not on the approved command list, so I'm not able to perform it.

If you'd like, I can instead:
- `cat shopkit/text.py` — show the contents of that file
- `ls` — list files in the current directory
- `git status` — show the repo's working tree status
- `python -m pytest -q` — run the test suite quietly

Let me know which of these you'd like, or rephrase your request if you meant something else.

User: app.ls

Agent selected: ls

Security: Command is ALLOWED!

AI Agent: The `ls` command executed successfully. Here's what's in the current directory:

- **`sample_data`** — a directory (or file) named `sample_data`

That's the only item present in the current working directory. Let me know if you'd like to do anything else, such as checking `g